# CenterPoint VoxelNet Architecture Walkthrough

This executable walkthrough follows the boundary-complete one-stage nuScenes VoxelNet assembly. It uses the tested package modules and a notebook-local tiny sparse backend because the production `SpMiddleResNetFHD` backend remains deliberately unimplemented; CUDA and `spconv` integration are deferred and unimplemented.

## Full Architecture

`points -> hard voxels -> mean VFE -> sparse backbone -> RPN neck -> six-task CenterHead -> loss or dense decode -> task NMS -> merged detections`

Input: batched LiDAR points with five features.
Output: six task prediction dictionaries or merged detections.
I/O evolution: `[N, 5] -> [M, 10, 5] -> [M, 5] -> [B, 256, H, W] -> [B, 512, H, W]`.
Test: `tests/test_voxelnet.py` traces feature shapes and gradients.

In [ ]:
import torch

from centerpoint import NUSCENES_VOXELNET_075
from centerpoint.contracts import TaskTargets, VoxelBatch
from centerpoint.data import HardVoxelizer
from centerpoint.models import (
    CenterHead,
    CenterPointPostprocessor,
    MeanVoxelFeatureEncoder,
    RPN,
    SparseBackbone,
    VoxelNet,
)
from centerpoint.ops import rotated_nms

torch.manual_seed(7)
config = NUSCENES_VOXELNET_075

## Coordinates And Tasks

LiDAR uses `x` forward, `y` left, and `z` up. Internal boxes are `[x, y, z, w, l, h, vx, vy, yaw]`; targets are `[dx, dy, z, log(w), log(l), log(h), vx, vy, sin(yaw), cos(yaw)]`. The six task groups are task-local class spaces.

Input: metric-space points and boxes.
Output: `zyx` sparse coordinates and task-local target maps.
I/O evolution: `xyz -> voxel xyz -> coordinate [batch, z, y, x] -> six heatmap/regression tasks`.
Test: `tests/test_contracts.py` and `tests/test_targets.py` establish these layouts.

In [ ]:
task_names = tuple(tuple(task) for task in config.tasks)
target_code_order = (
    "dx", "dy", "z", "log(w)", "log(l)",
    "log(h)", "vx", "vy", "sin(yaw)", "cos(yaw)",
)
assert len(task_names) == 6
assert len(target_code_order) == 10

## Data Preparation

A sweep record contributes `x, y, z, intensity, time_lag`; class-balanced sampling selects metadata with replacement; seeded global augmentation transforms points and boxes together; BEV filtering retains only in-range points and boxes.

Input: current and historical sweep records plus annotations.
Output: five-feature points and in-range internal boxes.
I/O evolution: `sweeps -> concatenated points -> augmented points/boxes -> prepared sample`.
Test: `tests/test_nuscenes_loading.py`, `tests/test_sampling.py`, and `tests/test_augmentation.py` cover the contracts.

In [ ]:
tiny_points = torch.tensor([
    [0.2, 0.2, 0.2, 0.5, 0.0],
    [1.1, 1.1, 1.1, 0.6, -0.1],
], dtype=torch.float32)
tiny_voxelizer = HardVoxelizer(
    voxel_size=(1.0, 1.0, 1.0),
    point_cloud_range=(0.0, 0.0, 0.0, 4.0, 4.0, 4.0),
    max_points_per_voxel=10,
    max_voxels=16,
)
tiny_voxels = tiny_voxelizer(tiny_points)
assert tiny_voxels.coordinates.tolist() == [[0, 0, 0], [1, 1, 1]]

## Hard Voxels And Mean VFE

Hard voxelization keeps first-occurrence voxel order, stores coordinates as `zyx`, and truncates points and voxels at configured limits. Mean VFE averages the valid points in each voxel.

Input: `[N, 5]` points.
Output: voxel features `[M, 5]` and coordinates `[M, 3]`.
I/O evolution: `[N, 5] -> [M, 10, 5] + [M] -> [M, 5]`.
Test: `tests/test_voxelization.py` verifies order, truncation, and mean VFE behavior.

In [ ]:
assert tiny_voxels.voxels.shape == (2, 10, 5)
assert tiny_voxels.num_points.tolist() == [1, 1]
assert tiny_voxels.coordinates.tolist() == [[0, 0, 0], [1, 1, 1]]

## Sparse Backbone Boundary

`SparseBackboneInput` separates package-level coordinate validation from the deferred sparse-convolution implementation. A production backend accepts mean features and `[batch, z, y, x]` coordinates, then emits dense BEV.

Input: `[M, 5]`, `[M, 4]`, spatial shape `(z, y, x)`, and batch size.
Output: dense BEV `[B, 256, H, W]`.
I/O evolution: `sparse records -> backend adapter -> dense NCHW BEV`.
Test: `tests/test_sparse_backbone_contract.py` validates shape, coordinate, dtype, and device boundaries.

In [ ]:
class TinySparseBackbone(SparseBackbone):
    """Notebook-only sparse boundary stand-in, not a production convolution backend."""

    def __init__(self):
        super().__init__(input_channels=5, output_channels=256, output_stride=8)

    def forward_sparse(self, inputs):
        height = inputs.spatial_shape[1] // self.output_stride
        width = inputs.spatial_shape[2] // self.output_stride
        bev = inputs.features.new_zeros((inputs.batch_size, 256, height, width))
        if inputs.features.shape[0]:
            values = inputs.features.sum(dim=1)
            batch, _, y, x = inputs.coordinates.long().unbind(dim=1)
            bev[batch, :, y // self.output_stride, x // self.output_stride] = values.unsqueeze(1)
        return bev

## RPN Neck

The tested RPN has two BEV blocks and matching deblocks. Their concatenation maps the sparse-backbone BEV feature to the CenterHead feature map.

Input: `[B, 256, H, W]` dense BEV.
Output: `[B, 512, H, W]` neck feature.
I/O evolution: `256 channels -> block/deblock 256 + block/deblock 256 -> 512 channels`.
Test: `tests/test_rpn_neck.py` covers shape, state layout, initialization, and gradients.

In [ ]:
neck = RPN(
    layer_nums=config.model.neck.layer_numbers,
    ds_layer_strides=config.model.neck.downsample_strides,
    ds_num_filters=config.model.neck.downsample_filters,
    us_layer_strides=config.model.neck.upsample_strides,
    us_num_filters=config.model.neck.upsample_filters,
    num_input_features=config.model.neck.input_channels,
)
assert neck(torch.zeros(1, 256, 4, 4)).shape == (1, 512, 4, 4)

## Six-Task CenterHead

A shared feature convolution feeds six task dictionaries. Each dictionary contains `hm`, `reg`, `height`, `dim`, `rot`, and `vel`; heatmap channel count is local to that task.

Input: `[B, 512, H, W]` shared BEV features.
Output: six branch dictionaries with NCHW maps.
I/O evolution: `shared 512-channel feature -> shared 64-channel feature -> six task-local heads`.
Test: `tests/test_center_head.py` checks task ordering, branches, shapes, losses, and gradients.

In [ ]:
head = CenterHead(
    in_channels=config.model.head.input_channels,
    tasks=config.tasks,
    common_heads={
        branch.name: (branch.output_channels, branch.num_convolutions)
        for branch in config.model.head.branches
    },
    share_conv_channel=config.model.head.shared_channels,
    loss_weight=config.model.head.loss_weight,
    code_weights=config.model.head.code_weights,
)
head_predictions, _ = head(torch.zeros(1, 512, 4, 4))
assert [prediction["hm"].shape[1] for prediction in head_predictions] == [1, 2, 2, 1, 2, 2]

## Losses

The heatmap focal loss supervises all center cells. Regression gathers each ten-code prediction at assigned indices, masks invalid objects, and applies the configured per-code weights.

Input: six prediction dictionaries and six `TaskTargets`.
Output: per-task heatmap and regression losses.
I/O evolution: `dense maps + indexed annotations -> gathered codes -> weighted task losses -> total`.
Test: `tests/test_losses.py` and `tests/test_center_head.py` verify focal, gathered regression, and loss composition.

In [ ]:
def tiny_targets(batch_size=1):
    targets = []
    for task in config.tasks:
        heatmap = torch.zeros((batch_size, len(task), 4, 4))
        heatmap[:, 0, 0, 0] = 1
        targets.append(TaskTargets(
            heatmap=heatmap,
            annotation=torch.zeros((batch_size, 1, 10)),
            indices=torch.zeros((batch_size, 1), dtype=torch.int64),
            mask=torch.ones((batch_size, 1), dtype=torch.uint8),
            categories=torch.zeros((batch_size, 1), dtype=torch.int64),
        ))
    return targets

## Dense Decode And NMS

The decoder evaluates every feature-map cell without top-K or local-maximum filtering, transforms offsets and regressions into metric boxes, runs task-wise rotated NMS, offsets local labels, and merges task results.

Input: six dense task prediction dictionaries.
Output: batch lists of `[x, y, z, w, l, h, vx, vy, yaw]` detections, scores, and global labels.
I/O evolution: `all cells -> metric candidates -> task NMS -> label offsets -> merged detections`.
Test: `tests/test_decoder.py`, `tests/test_rotated_nms.py`, and `tests/test_postprocess.py` cover the decode and merge contracts.

In [ ]:
postprocessor = CenterPointPostprocessor(
    config.make_decoder(),
    rotated_nms,
    config.tasks,
    iou_threshold=config.inference.nms.iou_threshold,
    pre_max_size=config.inference.nms.pre_max_size,
    post_max_size=config.inference.nms.post_max_size,
)
assert callable(postprocessor)

## End-To-End Tensor Ledger

The final cell injects `TinySparseBackbone`, builds `VoxelNet` from package modules, and runs forward features, loss/backward, and prediction on the intentionally small `(4, 32, 32)` spatial shape. This is an interface walkthrough, not evidence for a production sparse backend.

Input: four fixed hard voxels and six tiny target sets.
Output: compact stage shapes, gradient presence, and detection field shapes.
I/O evolution: `VoxelBatch -> six maps + [B, 512, 4, 4] -> losses/gradients -> detections`.
Test: `tests/test_voxelnet.py` provides the package-level end-to-end contract.

In [ ]:
model = VoxelNet(
    reader=MeanVoxelFeatureEncoder(config.model.num_input_features),
    backbone=TinySparseBackbone(),
    neck=neck,
    bbox_head=head,
    postprocessor=postprocessor,
    spatial_shape=(4, 32, 32),
)
voxels = VoxelBatch(
    voxels=torch.arange(200, dtype=torch.float32).reshape(4, 10, 5),
    num_points=torch.tensor([10, 8, 6, 4], dtype=torch.int32),
    coordinates=torch.tensor(
        [[0, 0, 0, 0], [0, 1, 8, 8], [0, 2, 16, 16], [0, 3, 24, 24]],
        dtype=torch.int32,
    ),
    batch_size=1,
)
predictions, neck_features = model.forward_features(voxels)
losses = model.loss(voxels, tiny_targets())
total_loss = sum(losses["loss"])
total_loss.backward()
model.eval()
detections = model.predict(voxels)

print("Tensor ledger")
print("neck", tuple(neck_features.shape))
print("heatmaps", [tuple(task["hm"].shape) for task in predictions])
print("gradient", model.neck.blocks[0][1].weight.grad is not None)
print("detection fields", tuple(detections[0].boxes.shape), tuple(detections[0].scores.shape), tuple(detections[0].labels.shape))